In [ ]:
import numpy as np
import pandas as pd
import os
import datetime as dt
import time
from copy import deepcopy
import matplotlib.pyplot as plt

from f1_elo.pvp.general import build_pvp_results
from f1_elo.pvp_model.dataset import get_model_ready_pvp_results
from f1_elo.pvp_model.gradient import (
    EloGradientOptimizer,
    split_dataset
)
from f1_elo.pvp_model.model import EloCalculation
from f1_elo.season import calc_season_drivers

import warnings
warnings.filterwarnings("ignore")

In [ ]:
build_pvp_results()
calc_season_drivers()
pvp_results = get_model_ready_pvp_results()

In [ ]:
settings = {
    'elo_game_value': 97.97,
    'num_rounds_degree': 0.0,
    'num_drivers_degree': 0.0,
    'calibrated_rating': 2000,
    'default_rating': 1830.9,
    'new_agent_alpha': 1.0,
    'saturation_rounds': 10,
}

elo_calc = EloCalculation(**settings)
elo_calc.run_pipeline(results=pvp_results)
elo_calc.log_loss

In [ ]:
# TODO

# 1. Run all round PVPs together instead of pair-by-pair update
# 2. Make abstract Elo calculation class

In [ ]:
datasets = split_dataset(df=pvp_results)

obj = EloGradientOptimizer()
output = obj.run(datasets=datasets, num_epochs=50)

In [ ]:
obj.optimized_parameters

In [ ]:
output

In [ ]:
col_name_mapping = {
    'elo_game_value': 'Elo game adjustment value',
    'num_rounds_degree': 'Degree for number of rounds adjustment for related season',
    'num_drivers_degree': 'Degree for number of drivers adjustment for related round',
    'default_rating': 'Initial rating for a new driver',
    'new_agent_alpha': 'Coefficient of new driver rating update increment',
}
titles = {
    'elo_game_value': 'Elo game correction coefficient per iteration',
    'num_rounds_degree': 'Degree for number of rounds correction coefficient per iteration',
    'num_drivers_degree': 'Degree for number of drivers correction coefficient per iteration',
    'default_rating': 'Initial rating for a new driver per iteration',
    'new_agent_alpha': 'Coefficient of new driver rating update increment per iteration',
}

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=3, figsize=(20, 8))
index = 0
axs = axs.flatten()
for col in ['elo_game_value', 'num_rounds_degree', 'num_drivers_degree', 'default_rating', 'new_agent_alpha']:
    axs[index].plot(output['iteration'], output[col])
    axs[index].set_xlabel('iteration')
    axs[index].set_ylabel(col_name_mapping.get(col))
    axs[index].set_title(titles.get(col))
    index += 1
# plt.tight_layout(pad=0.5, w_pad=0.5, h_pad=5.0)